# End-to-end walkthrough — Module A export pipeline

**Objective:** Trace how one **entity** moves from synthetic generation through cleaning, feature construction, segmentation (k=6), and participation propensity scoring.

**Determinism:** Export fixes RNG seeds to **42** for generation, flaw injection, cleaning, segmentation, and propensity (`model_run_manifest.json` lists the same).

**Runtime:** A full `run_export` at `sample_size=15_000` mirrors CI stability for department rake gates (several minutes on a laptop). Reduce only for exploratory debugging—expect calibration QA failures below ~10k rows.

**Canonical CLI:** `poetry run python -m population_segmentation.pipeline` from the repository root (defaults documented in `population_segmentation.pipeline.__main__`).

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import yaml

# Repository root = cwd when Jupyter is launched from repo root
ROOT = Path.cwd().resolve()
CFG = ROOT / "module_a_population_segmentation/config/generation.yaml"
ANCH = ROOT / "module_a_population_segmentation/config/calibration_anchors.yaml"
OUT = ROOT / "data/processed/notebook_walkthrough"

assert CFG.is_file(), f"Run from repo root; missing {CFG}"
assert ANCH.is_file(), f"Missing {ANCH}"

with open(CFG, encoding="utf-8") as f:
    config = yaml.safe_load(f)
with open(ANCH, encoding="utf-8") as f:
    anchors = yaml.safe_load(f)

from population_segmentation.pipeline.export import run_export

artifacts = run_export(config, anchors, OUT, sample_size=15_000)
print("Artifacts:", {k: str(v) for k, v in artifacts.items()})

manifest = json.loads(artifacts["model_run_manifest"].read_text(encoding="utf-8"))
assert manifest["random_seeds"]["propensity"] == 42

In [ ]:
master = pd.read_parquet(artifacts["population_master_clean"])
labels = pd.read_parquet(artifacts["segment_labels"])
prop = pd.read_parquet(artifacts["participation_propensity"])

entity_id = int(master["entity_id"].iloc[0])
row_m = master.loc[master["entity_id"] == entity_id].iloc[0]
row_l = labels.loc[labels["entity_id"] == entity_id].iloc[0]
row_p = prop.loc[prop["entity_id"] == entity_id].iloc[0]

print("entity_id", entity_id)
print("department", row_m["department"], "segment", row_l["segment_label"], "propensity", row_p["participation_propensity"])